In [ ]:
# Example 1: get_cpu_info() returns a dict describing the current CPU.
import cpuinfo

info = cpuinfo.get_cpu_info()
print(f"type: {type(info).__name__}, {len(info)} keys")
for key in ('arch', 'bits', 'count', 'arch_string_raw'):
    print(f"  {key}: {info.get(key)!r}")

assert isinstance(info, dict)
# Core keys are always present regardless of platform.
for key in ('arch', 'bits', 'count'):
    assert key in info, f"missing expected key {key!r}"
assert isinstance(info['count'], int) and info['count'] >= 1
assert info['bits'] in (32, 64)

In [ ]:
# Example 2: get_cpu_info_json() returns a JSON string that round-trips.
import json

import cpuinfo

blob = cpuinfo.get_cpu_info_json()
print(f"JSON length: {len(blob)} bytes")
parsed = json.loads(blob)

print(f"parsed arch: {parsed.get('arch')!r}")
assert isinstance(parsed, dict)
# The JSON view carries the same core fields as the dict view.
assert parsed['arch'] == cpuinfo.get_cpu_info()['arch']
assert parsed['bits'] == cpuinfo.get_cpu_info()['bits']

In [ ]:
# Example 3: module-level version metadata + public callables are present.
import cpuinfo

print(f"CPUINFO_VERSION: {cpuinfo.CPUINFO_VERSION}")
print(f"CPUINFO_VERSION_STRING: {cpuinfo.CPUINFO_VERSION_STRING}")

# Version metadata is a (major, minor, patch) tuple.
assert isinstance(cpuinfo.CPUINFO_VERSION, tuple) and len(cpuinfo.CPUINFO_VERSION) == 3

# The documented entry points are all callable.
for name in ('get_cpu_info', 'get_cpu_info_json'):
    assert callable(getattr(cpuinfo, name)), f"cpuinfo.{name} not callable"
print("get_cpu_info / get_cpu_info_json are callable")

In [ ]:
# Example 4: CPU flags surface — a list of feature strings.
import cpuinfo

info = cpuinfo.get_cpu_info()
flags = info.get('flags', [])
print(f"reported {len(flags)} CPU flags")
print(f"first 12: {flags[:12]}")

assert isinstance(flags, list)
# Flags are lowercase feature tokens; the list is de-duplicated + sorted by the lib.
assert all(isinstance(f, str) for f in flags)
assert flags == sorted(flags)

In [ ]:
# Example 5: results are deterministic across repeated calls (cache/probe stability).
import cpuinfo

first = cpuinfo.get_cpu_info()
second = cpuinfo.get_cpu_info()

# Identity fields must not drift between probes on the same machine.
for key in ('arch', 'bits', 'count', 'arch_string_raw', 'vendor_id_raw'):
    assert first.get(key) == second.get(key), f"{key} drifted between calls"

print(f"arch={first['arch']} bits={first['bits']} count={first['count']}")
print("py_cpuinfo smoke test PASS")